In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "../input/ventilator-pressure-prediction"
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sample_path)


def add_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["step"] = (
        df.groupby("breath_id", sort=False).cumcount().astype(np.int16)
    )  # 0..79

    df["u_in_r1"] = np.rint(df["u_in"].to_numpy() * 10.0).astype(np.int16)
    df["u_in_b05"] = np.rint(df["u_in"].to_numpy() * 2.0).astype(np.int16)

    u_in_diff = (
        df.groupby("breath_id", sort=False)["u_in"].diff().fillna(0.0).to_numpy()
    )
    df["du_in_b05"] = np.rint(u_in_diff * 2.0).astype(np.int16)

    u_in_cum = df.groupby("breath_id", sort=False)["u_in"].cumsum().to_numpy()
    df["u_in_cum_b01"] = np.rint(u_in_cum * 1.0).astype(np.int32)  # 1.0 resolution

    u_in_lag1 = (
        df.groupby("breath_id", sort=False)["u_in"].shift(1).fillna(0.0).to_numpy()
    )
    df["u_in_lag1_b05"] = np.rint(u_in_lag1 * 2.0).astype(np.int16)

    u_out_lag1 = (
        df.groupby("breath_id", sort=False)["u_out"].shift(1).fillna(0).to_numpy()
    )
    df["u_out_lag1"] = u_out_lag1.astype(np.int8)

    return df


train_k = add_keys(train)
test_k = add_keys(test)

keys_full = ["R", "C", "u_out", "step", "u_in_r1"]
keys_full_b05 = ["R", "C", "u_out", "step", "u_in_b05"]
keys_full_b05_du = ["R", "C", "u_out", "step", "u_in_b05", "du_in_b05"]

keys_full_b05_cum = ["R", "C", "u_out", "step", "u_in_b05", "u_in_cum_b01"]
keys_wo_uout_b05_cum = ["R", "C", "step", "u_in_b05", "u_in_cum_b01"]

keys_full_b05_lag = [
    "R",
    "C",
    "u_out",
    "step",
    "u_in_b05",
    "u_in_lag1_b05",
    "u_out_lag1",
]

keys_wo_uin = ["R", "C", "u_out", "step"]
keys_wo_uout = ["R", "C", "step", "u_in_r1"]
keys_wo_uout_b05 = ["R", "C", "step", "u_in_b05"]
keys_wo_uout_b05_du = ["R", "C", "step", "u_in_b05", "du_in_b05"]

keys_rc_step = ["R", "C", "step"]
keys_rc = ["R", "C"]

train_insp = train_k[train_k["u_out"] == 0].copy()

mean_rc_step = train_insp.groupby(keys_rc_step, observed=True)["pressure"].mean()
mean_rc = train_insp.groupby(keys_rc, observed=True)["pressure"].mean()
global_mean = float(train_insp["pressure"].mean())


def smoothed_group_mean(
    df: pd.DataFrame,
    fine_keys: list,
    prior_series: pd.Series,
    prior_keys: list,
    alpha: float,
) -> pd.Series:
    g = df.groupby(fine_keys, observed=True)["pressure"]
    mu = g.mean()
    cnt = g.size().astype(np.float32)

    fine_index_df = mu.index.to_frame(index=False)
    prior_map = prior_series.to_dict()
    prior_tuples = list(fine_index_df[prior_keys].itertuples(index=False, name=None))
    prior_vals = np.fromiter(
        (prior_map.get(t, global_mean) for t in prior_tuples),
        dtype=np.float32,
        count=len(prior_tuples),
    )

    cnt_vals = cnt.reindex(mu.index).to_numpy(dtype=np.float32, copy=False)
    smooth_vals = (
        cnt_vals * mu.to_numpy(dtype=np.float32, copy=False) + alpha * prior_vals
    ) / (cnt_vals + alpha)
    return pd.Series(smooth_vals, index=mu.index)


ALPHA_FULL = 10.0
ALPHA_MID = 20.0

mean_wo_uin = train_insp.groupby(keys_wo_uin, observed=True)["pressure"].mean()

mean_full = smoothed_group_mean(
    train_insp, keys_full, mean_wo_uin, keys_wo_uin, ALPHA_FULL
)
mean_full_b05 = smoothed_group_mean(
    train_insp, keys_full_b05, mean_wo_uin, keys_wo_uin, ALPHA_FULL
)

prior_full_b05 = mean_full_b05

mean_full_b05_du = smoothed_group_mean(
    train_insp, keys_full_b05_du, prior_full_b05, keys_full_b05, ALPHA_MID
)
mean_full_b05_cum = smoothed_group_mean(
    train_insp, keys_full_b05_cum, prior_full_b05, keys_full_b05, ALPHA_MID
)
mean_full_b05_lag = smoothed_group_mean(
    train_insp, keys_full_b05_lag, prior_full_b05, keys_full_b05, ALPHA_MID
)

mean_wo_uout_b05_cum = train_insp.groupby(keys_wo_uout_b05_cum, observed=True)[
    "pressure"
].mean()
mean_wo_uout = train_insp.groupby(keys_wo_uout, observed=True)["pressure"].mean()
mean_wo_uout_b05 = train_insp.groupby(keys_wo_uout_b05, observed=True)[
    "pressure"
].mean()
mean_wo_uout_b05_du = train_insp.groupby(keys_wo_uout_b05_du, observed=True)[
    "pressure"
].mean()

# Change (metric-aligned, minimal): build expiratory-only mean tables for u_out==1 rows,
# so expiratory predictions don't rely on inspiratory-only statistics.
train_exp = train_k[train_k["u_out"] == 1].copy()
exp_global_mean = float(train_exp["pressure"].mean()) if len(train_exp) else global_mean
exp_mean_rc_step = (
    train_exp.groupby(keys_rc_step, observed=True)["pressure"].mean()
    if len(train_exp)
    else mean_rc_step
)
exp_mean_rc = (
    train_exp.groupby(keys_rc, observed=True)["pressure"].mean()
    if len(train_exp)
    else mean_rc
)

test_full = list(test_k[keys_full].itertuples(index=False, name=None))
test_full_b05 = list(test_k[keys_full_b05].itertuples(index=False, name=None))
test_full_b05_du = list(test_k[keys_full_b05_du].itertuples(index=False, name=None))
test_full_b05_cum = list(test_k[keys_full_b05_cum].itertuples(index=False, name=None))
test_full_b05_lag = list(test_k[keys_full_b05_lag].itertuples(index=False, name=None))

test_wo_uout_b05_cum = list(
    test_k[keys_wo_uout_b05_cum].itertuples(index=False, name=None)
)
test_wo_uin = list(test_k[keys_wo_uin].itertuples(index=False, name=None))
test_wo_uout = list(test_k[keys_wo_uout].itertuples(index=False, name=None))
test_wo_uout_b05 = list(test_k[keys_wo_uout_b05].itertuples(index=False, name=None))
test_wo_uout_b05_du = list(
    test_k[keys_wo_uout_b05_du].itertuples(index=False, name=None)
)

test_rc_step = list(test_k[keys_rc_step].itertuples(index=False, name=None))
test_rc = list(test_k[keys_rc].itertuples(index=False, name=None))

pred = np.empty(len(test_k), dtype=np.float32)
u_out_arr = test_k["u_out"].to_numpy()

for i in range(len(test_k)):
    if u_out_arr[i] == 1:
        # Change: use expiratory-only tables for expiratory rows.
        v = exp_mean_rc_step.get(test_rc_step[i], np.nan)
        if pd.isna(v):
            v = exp_mean_rc.get(test_rc[i], exp_global_mean)
        pred[i] = float(v)
        continue

    v = mean_full_b05_lag.get(test_full_b05_lag[i], np.nan)
    if pd.isna(v):
        v = mean_full_b05_cum.get(test_full_b05_cum[i], np.nan)
    if pd.isna(v):
        v = mean_full_b05_du.get(test_full_b05_du[i], np.nan)
    if pd.isna(v):
        v = mean_full.get(test_full[i], np.nan)
    if pd.isna(v):
        v = mean_full_b05.get(test_full_b05[i], np.nan)
    if pd.isna(v):
        v = mean_wo_uin.get(test_wo_uin[i], np.nan)
    if pd.isna(v):
        v = mean_wo_uout_b05_cum.get(test_wo_uout_b05_cum[i], np.nan)
    if pd.isna(v):
        v = mean_wo_uout.get(test_wo_uout[i], np.nan)
    if pd.isna(v):
        v = mean_wo_uout_b05_du.get(test_wo_uout_b05_du[i], np.nan)
    if pd.isna(v):
        v = mean_wo_uout_b05.get(test_wo_uout_b05[i], np.nan)
    if pd.isna(v):
        v = mean_rc_step.get(test_rc_step[i], np.nan)
    if pd.isna(v):
        v = mean_rc.get(test_rc[i], global_mean)

    pred[i] = float(v)

pressure_levels = np.sort(train["pressure"].unique()).astype(np.float32)
mids = (pressure_levels[:-1] + pressure_levels[1:]) * 0.5
idx = np.searchsorted(mids, pred, side="left")
pred = pressure_levels[idx]



In [2]:
sub_out = test[["id"]].copy()
sub_out["pressure"] = pred
sub_out.to_csv("submission.csv", index=False)

print(sub_out.head())
print(sub_out.shape)
print(sub_out.isna().sum())
print("Saved submission.csv")

   id   pressure
0   1   6.399909
1   2   5.837492
2   3   6.892024
3   4  10.969548
4   5  19.335505
(603600, 2)
id          0
pressure    0
dtype: int64
Saved submission.csv
